In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
client = chromadb.PersistentClient(path="./chromadb")

def chunk_with_overlap(text: str, chunk_size: int = 500, overlap: int = 150) -> list[str]:
    chunks = []
    step = chunk_size - overlap
    for i in range (0, len(text), step):
        chunk = text[i : i + chunk_size]
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

try:
    client.delete_collection('rag_kb')
except Exception:
    pass
collection = client.create_collection(
    name='rag_kb',
    metadata={"hnsw:space": "cosine"}

)
for fname in os.listdir('data'):
    with open(os.path.join('data',fname), 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = chunk_with_overlap(text)
    emb = model.encode(chunks)
    collection.upsert(
        documents = chunks,
        embeddings = emb.tolist(),
        ids=[f"{fname}_{i}" for i in range(len(chunks))],
        metadatas=[{'source': fname} for _ in chunks]
)
print(f"{fname}: {len(chunks)} чанков")

res = collection.get(where={'source': 'transformer_notes.txt'})
print("чанков из transformer_notes:", len(res['ids']))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5700.92it/s]


transformer_notes.txt: 15 чанков
чанков из transformer_notes: 15


: 

In [3]:
import os
print(os.listdir('data'))

['article.txt', 'transformer_notes.txt']


In [1]:
import os
for fname in os.listdir('data'):
    with open(os.path.join('data', fname), 'r', encoding='utf-8') as f:
        t = f.read()
    print(repr(fname), len(t))

'article.txt' 15069
'transformer_notes.txt' 5216
